# 07 · Results

**What this notebook does.** It makes every result figure and test of the project from the data computed in 06 (and the tables of 01–05).
Nothing heavy is computed here, except a few cells that compute something the first time and then re-use a saved file (they say so in their method text).

**Run order in the project:** 01 → 02 → 03 → 04 → 05 → 06 → **07**.

**How it works (important):**
1. Run the **settings cell** (it is the only cell you change).
2. Then run **any** cell, in any order. Every cell is one line: `fxr.run(<file>.<cell name>)`.
   If a cell needs a result of an earlier cell (e.g. Step 5 figure needs Step 5), the earlier cell is run first automatically, silently
   (no figure, no print, nothing saved). So each cell can run right after the settings cell.
3. The code of each cell is a function with the same name in one of four files: `fx_res_power.py` (section 1 + theta along the session),
   `fx_res_pilot.py` (pilot of the coupling), `fx_res_coupling.py` (coupling), `fx_res_bursts.py` (bursts + bursts along the session).
   The engine (`run`, saving, settings) is `fx_results.py`; colours and saving are in `fx_common.py`.
4. Numbers that belong to one analysis only (minimum trials, windows, number of shuffles) are written at the top of that function, with a comment.
   To change one: open the file, change the number, save, then `importlib.reload(<file>)` (or restart the kernel) and run the cell again.

**Sections** (colour of a heading = frequency band)

| Section | Question | Main cells |
|---|---|---|
| 1. Power during the hold | Does theta rise while the rat holds the lever? Is delta about the action (ERP)? Does high gamma / firing drop? | M1, 1c–1n, total vs induced, MUA |
| 2. Phase coupling | Are IL and PL theta phases locked during the hold, in every outcome, with IL's phase ahead? Jäckel replication | pilot, A1–A2d, T4–T15, T8b/T8c |
| 3. Beta bursts | More beta bursts before late responses? Burst timing vs RT, after errors, burst shape; gamma as control | Steps 1–2, 5–10 |
| 4. Along the session | How the session is split; theta, coupling and bursts at the start / middle / end; RT vs theta drop | S13, coupling along the session, Steps 0, 3, 4, Gamma Step 1 |

**Colours.** Headings: theta green, beta / bursts blue, gamma purple, delta amber. Figures: correct green, late orange, early yellow;
short vs long of the same outcome: long green, short red (wait = long, go = short); rats: three pinks (Rat 1 light, Rat 2 medium, Rat 3 dark);
bands: theta green, delta amber, gamma purple, beta blue; regions: IL orange, PL blue, Cg1 violet.

**Statistics used (details in each cell):** session = unit; sign-flip test across sessions (`fn.sign_flip_p`, 10,000 flips); for the coupling:
`strat_test` = sign-flip with every rat weighted equally; Rayleigh test for directions; mixed models (statsmodels) with a random intercept per session for
trial-level burst analyses; FDR = Benjamini–Hochberg where stated; otherwise raw p.

**Removed** (not used by any result, duplicates or checks): 1a (test session), 1e-check, 1e-check2, "why 44 sessions", the report-renumber and
save-name check cells, Step 6 (old version with block) and 6b (IL vs PL check), the second copy of 1m-delta, the "equal n" helper and the Fig. 15
model cells (their input cells are missing; Gamma Step 1 keeps its own figure), the STATUS / SUMMARY / T3 cells of the coupling.

## Settings

| Setting | What it is | Change when |
|---|---|---|
| `CODE_DIR` | folder with the notebooks, `functions.py` and the `fx_*.py` files | the repository is somewhere else |
| `DATA_ROOT` | project data folder (contains `amir_data`, `2_processed`) | the server path changes |
| `TABLES` | tables of 01–05 (`final_sessions.csv`, `trials_filtered.csv`, ...) — **temporarily** the old folder in the code folder | when the tables move to `amir_data\tables` |
| `FIG_DIR` | where figures go when saving is on (one sub-folder per cell) | you want them elsewhere |
| `OUT_TABLES` | where result tables go when saving is on (one sub-folder per cell) | you want them elsewhere |
| `SAVE` | `False` = nothing is written (figures only shown). `True` = every cell writes its figures (PNG) and tables | to save everything |
| `A_RUN` | which run of A1–A2d: 1 = SR30-500, press → +1 s (main) · 2 = CMR, press → +1 s · 3 = SR30-500, press → release · 4 = CMR, press → release | to see the other reference / window |

**Saving one cell only, to a place you choose** (also when `SAVE = False`): write the folder in that cell's run line, e.g.
`fxr.run(fxp.theta_all_sessions, save_to=r"C:\Users\ar687\Desktop\slides")`.

**Files that are always written:** cells that compute something the first time keep a data file (rows / cache) in the data folders, so the next run is fast.
They never overwrite a finished session.

**Re-running:** `fxr.run(cell, again=True)` also re-runs the earlier cells it needs; `fxr.forget()` forgets all results of this kernel.

In [ ]:
import os, sys, importlib
CODE_DIR   = r"C:\Users\ar687\Documents\GitHub\AmirAnalysis\main"          # folder of the notebooks and fx_*.py files
DATA_ROOT  = r"O:\archive\projects\2020_euSNN\Neuropixels_RT_mPFC\data"      # project data (amir_data, 2_processed inside)
TABLES     = os.path.join(CODE_DIR, "tables")                                 # TEMPORARY: tables of 01–05 (later os.path.join(DATA_ROOT, "amir_data", "tables"))
FIG_DIR    = os.path.join(DATA_ROOT, "amir_data", "figures", "07_results")    # figures, if SAVE (one sub-folder per cell)
OUT_TABLES = os.path.join(DATA_ROOT, "amir_data", "tables", "07_results")     # result tables, if SAVE (one sub-folder per cell)
SAVE       = False                                                            # True = write figures (PNG) and tables
A_RUN      = 1                                                                # A1–A2d: 1 SR30-500 press→1 s | 2 CMR press→1 s | 3 SR30-500 press→release | 4 CMR press→release

sys.path.insert(0, CODE_DIR)
import fx_results as fxr, fx_res_power as fxp, fx_res_pilot as fxpi, fx_res_coupling as fxco, fx_res_bursts as fxb
fxr.setup(CODE_DIR, DATA_ROOT, TABLES, FIG_DIR, OUT_TABLES, SAVE=SAVE, A_RUN=A_RUN)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

## 1. Power during the hold
While the rat holds the lever (press → tone), does the power of the mPFC LFP change? **Theta** (4–7 Hz) first (M1 and 1c–1n), then
**delta** (1–3 Hz) with and without the evoked potential (ERP), then **high gamma** (61–100 Hz) together with the **multi-unit firing**.
Band power comes from 06 step 1 (Morlet wavelets; window value = log10 of the mean power in the window; curves = log10(power(t) / own pre-press power),
each trial only while the paw is on the lever). Windows: pre −0.5…−0.2 s (press) · hold 0.20–0.85 s (press) · post 0.2–0.5 s (release) ·
pre_tone / post_tone 0.3 s before / after the tone. References: CMR and LOCAL (every theta result is shown with both).

</div>

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Theta

</div>

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 1b. M1 on all finished sessions (CMR)
The same value for every finished session (>= 10 correct long trials) -> table M1 (one row per session × band).
Sign-flip test across sessions (fn.sign_flip_p, 10,000 flips): is the mean over sessions different from 0? Per rat and all rats;
raw p and FDR q (per rat: 6 bands × 2 comparisons = 12 tests). p floor 0.0001 -> reported as p < 0.001.
Result (58 of 60 sessions; 2 with < 10 correct long trials: 135_0510_1625, 134_0620_1512): theta hold-pre +23.9 % (58/58),
hold-post +55.4 % (57/58); alpha +16.5 / +27.4 %; beta +12.6 / +20.0 %; delta +9.9 % (40/58) / +33.3 %; high gamma −5.5 / −5.8 %;
all q < 0.001.

**Why:** the first question of the project: is there more theta (and other bands) while the rat holds the lever than before the press and after
the release? The session is the unit (trials of one session are not independent); the sign-flip test needs no normal distribution.

</div>

In [ ]:
fxr.run(fxp.m1_all_sessions)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 1c. Pilot: session 20260529_r0000136_wt_1318, comparison example (how the band power is computed)
Same session, same 20 channels (fn.select_channels, n_total = 20) and same 49 correct long trials as the pilot; CMR.
Needs 1e-20 and 1e-20b (Morlet power of the 20 channels around the press and the release) run directly before this cell.
Per trial and channel: Morlet wavelet power over time (same frequencies and cycles as 06: theta 4–7 Hz, 5 cycles),
cut around the press (−2.5..3.5 s) and around the release (−3.0..2.5 s).
Windows: pre −0.5..−0.2 s (press), hold 0.20..0.85 s (press), post 0.2..0.5 s (release).
Per window: log10 of the mean power over the band frequencies and the window time -> hold − pre and hold − post (log ratio),
mean over trials, then mean over the 20 channels; shown as % change.
Heat maps (display only; the numbers come from the windows): log10 power minus each trial's own baseline, mean over trials and channels:
pilot_comparison.png (pre | hold | post): baseline = mean of own pre and post (the hold colour lies between hold − pre and hold − post);
pilot_hold_vs_pre.png: baseline = own pre-press; pilot_hold_vs_post.png: baseline = own post-release.
Check: recomputed theta hold − pre = saved value (+31.9 % mean, +31.1 % median over the 20 channels; range +5 to +65 %, all channels up).
The middle of the hold is cut in the joined plots because the hold length differs between trials.

**Why:** shows on one session, step by step, how a window value is computed (the same computation as 06), before the numbers of all sessions.

</div>

In [ ]:
fxr.run(fxp.pilot_time_frequency)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 1d + 1f. Pilot session: theta over time and significance per channel (one cell)
This cell runs the two old pilot cells together (computed as before):

**1f — theta power across time (look at the data first).** Raw theta power (log10 of the mean Morlet power over 4, 5, 6, 7 Hz, 5 cycles; a.u.)
over time, no baseline; mean over the 49 correct long trials, then over the 20 selected channels (fn.select_channels); test session
20260529_r0000136_wt_1318, CMR. Left: around the press (pre and hold windows shaded); right: around the release (post shaded). No test.

**1d — significance per channel.** Same session, 20 channels, 49 correct long trials, CMR. Per channel and band: hold − pre and hold − post
(log10 ratio per trial); sign-flip test across TRIALS (5000 flips). * = p < 0.05 vs before the press AND vs after the release, same direction
(hold-specific); · = p < 0.05 in one panel only. Colour = % change (geometric mean over trials); white frame = theta.
Result (raw p): theta hold-specific in 12/20 channels, alpha 6/20, delta 1/20, beta 1/20, low γ 2/20, high γ 1/20 (about 1/20 expected by chance
per band; channels are not independent, so this is only a guide). Probe plots: the 20 selected channels and the theta-significant ones.

**Why:** the pilot gave the first sign that theta rises while the rat holds the lever; it is then tested across sessions (1b, 1e).

</div>

In [ ]:
fxr.run(fxp.pilot_theta_over_time)
fxr.run(fxp.pilot_significance)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 1e. Theta across all sessions (correct long), CMR and LOCAL
- Same figure for both references; q = FDR over 12 tests (6 bands x 2 comparisons, all rats), as in 1b.
- Result: CMR hold vs pre +23.9 % (58/58 up), hold vs post +55.4 %; LOCAL +13.0 % (56/58), +20.1 %; all q < 0.001.

**Test:** sign-flip across sessions (10,000 flips) of the session values (hold − pre, hold − post), FDR over 6 bands × 2 comparisons.
**Why:** the main theta figure (report Fig. 2D): one value per session, both references, so the result does not depend on the reference.

</div>

In [ ]:
fxr.run(fxp.theta_all_sessions)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 1e-b. The Fig. 2D numbers for each outcome alone (correct, late, early), CMR and LOCAL
**What:** the same two measures as 1e (hold vs before the press, hold vs after the release) computed separately for correct, late and early long trials.
**Computation:** per trial and channel log10(hold / pre) and log10(hold / post); mean over trials, median over channels → one value per session and outcome.
**Test:** sign-flip across sessions for each outcome and measure; FDR (Benjamini–Hochberg) over the tests of one reference. Numbers only (no figure).
**Why:** to check that the theta rise during the hold is not only a property of correct trials.

</div>

In [ ]:
fxr.run(fxp.theta_each_condition_numbers)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 1g. Theta time course in all long trials: correct, late, early (all sessions, all rats)
Long trials only (tone at 1.05 s), so the windows mean the same in every condition. Curves from the band-power files (06),
each trial only while the paw is on the lever. Per session: condition with >= 10 trials (correct long) or >= 5 (late long,
early long held > 0.3 s); median over channels; time point kept if >= 50 % of the trials are on the lever.
Across sessions: mean ± SEM (session = unit); time point drawn if >= 50 % of the sessions have a value. No tests.
- Figure 1: theta power across time, no baseline (log10, a.u.; rebuilt as change curve + the condition's own pre-press level,
because raw curves were not saved in 06 — same shape as Figure 2).
- Figure 2: hold and after the release, both compared to before the press (%).
- Figure 3: hold compared to before the press (left, pre = 0) and hold compared to after the release (right, post = 0; %).Result (CMR): the same rise in all outcomes, peak about 0.55 s after the press (correct +21 %, late +20 %, early +26 %);
hold vs pre [ ] / [ ] / [ ] %, hold vs post [ ] / [ ] / [ ] % (from the printout).
theta_hold_vs_pre_and_post_long_<ref>.png

**Why:** shows the whole time course before any window is chosen; the windows of 1e lie where the curves differ.

</div>

In [ ]:
fxr.run(fxp.theta_time_course_long)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 1h. Theta during the hold, all long trials pooled (correct + late + early held >= 0.85 s), CMR and LOCAL
Same as 1e, but per session all long trials whose hold window lies completely on the lever are pooled:
all correct long, all late long, and early long trials held >= 0.85 s (the stored hold window 0.20–0.85 s is not cut at the release,
so shorter early trials are left out). Per channel: mean over trials of log10(hold / pre) and log10(hold / post); median over channels
-> one value per session (>= 10 pooled trials). One dot = one session (colour = rat), black = mean ± SEM over sessions.
Sign-flip test across sessions (10,000 flips), raw p (p floor 0.0001 -> reported as p < 0.001).
Question: is the theta rise during the hold present across all long trials, not only correct ones (the rise is the same in all outcomes, 1g)?

**Why:** a check that the rise is a property of holding, not of the trial outcome.

</div>

In [ ]:
fxr.run(fxp.theta_all_long_pooled)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 1i. Theta during the hold per region (PL, IL, Cg1), CMR and LOCAL (personal note)
Exploratory only (no region hypothesis) — not for the report.
Correct long trials (>= 10 per session). Per channel: mean over trials of log10(hold / pre) and log10(hold / post);
per region (channel_regions_all.csv, fn.region_masks; >= 10 channels in the region): median over the region's channels
-> one value per session and region. Sign-flip across sessions (10,000 flips): each region vs 0, and paired region differences
(IL − PL, IL − Cg1, PL − Cg1; sessions with both regions). Per rat: mean % only.
FDR (Benjamini–Hochberg) per reference: regions 3 × 2 comparisons = 6 tests; differences 3 pairs × 2 comparisons = 6 tests -> q.
Paper (Fakhraei 2021): PL (A32D) only as one SVM feature; IL not reported; no region comparison.
amir_data\Theta\across regions\theta_regions_CMR.png, theta_regions_LOCAL.png

**Why:** exploratory: is the theta rise carried by one mPFC region?

</div>

In [ ]:
fxr.run(fxp.theta_per_region)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 1r. All theta findings per region (IL, PL, Cg1): does any finding differ between regions? CMR and LOCAL
**What:** every theta finding of this section (hold vs pre, wait − go, drop after the release, outcome differences) computed per region.
**Computation:** measures 1, 3, 4, 5 from the band-power files (per channel mean over trials, median over the region's channels → one value per session
and region); measure 2 (wait − go) from the theta signal of all channels (06 step 8), because go trials continue after their release.
**Test:** each region vs 0 (sign-flip across sessions), and region pairs paired within session (sign-flip); Spearman where a correlation is needed;
FDR per reference. Figure: one panel per measure, regions with 95 % CI, brackets for region pairs.
**Why:** to know whether the theta results are mPFC-wide or specific to IL / PL.

</div>

In [ ]:
fxr.run(fxp.theta_findings_per_region)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 1j. Theta drops at the release, before the first lick (CMR and LOCAL)
Question: is the drop of theta after the release caused by licking?
A: per session, long trials of each outcome (>= 5): correct, late, early held >= 0.85 s (hold window fully on the lever);
per channel mean over trials of log10(post / hold); median over channels -> one value per session.
Sign-flip across sessions (10,000 flips): each outcome vs 0, and late − correct, early − correct (paired); FDR over the 5 tests -> q.
Shown as post vs hold (negative = drop; the same number as hold vs post with the sign flipped).
B: first lick after the release (tables\licks_after_release_long.csv), per outcome; the theta post window (0.2–0.5 s) shaded.
If the first lick comes after the post window, the drop starts before licking.
LOCAL reference: with CMR, late − correct differs; with LOCAL it does not.
Result (LOCAL): drop in correct [ ] %, late [ ] %, early [ ] %; late − correct q = [ ], early − correct q = [ ];
first lick median [ ] s after the release.

**Why:** licking produces movement artefacts and rhythmic activity; if the drop starts before the first lick, licking cannot explain it.

</div>

In [ ]:
fxr.run(fxp.theta_drop_before_lick)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 1j-b. First lick after the release vs the theta window after the release (0.2–0.5 s), long trials
**What:** when does the first lick happen relative to the post-release window used for theta?
**Computation:** first lick time after the release for every long trial (tables\licks_after_release_long.csv), per outcome; distribution shown with the
theta post window (0.2–0.5 s) shaded. No test (descriptive).
**Why:** companion of 1j: if most first licks come after 0.5 s, the theta drop measured in 0.2–0.5 s is not a licking effect.

</div>

In [ ]:
fxr.run(fxp.first_lick_vs_window)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 1k. Is the theta drop after the release explained by movement? (high gamma as movement proxy), CMR and LOCAL
Correct long trials, stored windows. Per trial (median over channels): theta drop = theta post − hold; movement proxy =
high gamma (61–100 Hz) post − hold (high gamma rises with movement / muscle activity).
Per session (>= 10 trials): (1) Spearman rho theta drop vs high-gamma change; (2) linear fit theta drop ~ high-gamma change ->
intercept = theta drop expected with no high-gamma change; (3) theta drop in the lowest and highest third of high-gamma change.
Across sessions: sign-flip test (10,000 flips; rho vs 0, intercept vs 0, thirds vs 0, highest − lowest); FDR over the 7 tests per reference -> q.
If the intercept and the lowest third still show a clear theta drop -> the drop is not explained by movement.
Figure: A theta drop (all trials, at zero high-gamma change, lowest and highest third); B rho per session.
Result: CMR intercept [ ] %, lowest third [ ] %, rho [ ]; LOCAL intercept [ ] % (earlier run −16.6 %), lowest third [ ] %, rho [ ].
amir_data\Theta\movement check\theta_drop_vs_movement_CMR.png, theta_drop_vs_movement_LOCAL.png, theta_drop_vs_movement_stats_CMR_LOCAL.csv

**Why:** movement after the release could change theta; high gamma is used as a movement proxy because it rises with muscle activity.

</div>

In [ ]:
fxr.run(fxp.theta_drop_vs_movement)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Fig. S12. Theta drop after the release with and without movement (high gamma as proxy), correct long, CMR
**What:** the report figure of 1k.
**Computation:** per trial theta drop (post − hold) and high-gamma change (post − hold), median over channels; per session the drop at zero high-gamma
change (intercept of the line) and in the lowest / highest third of the high-gamma change. Figure only, the tests are in 1k.
**Why:** one picture of "the theta drop is still there when there is little movement".

</div>

In [ ]:
fxr.run(fxp.theta_drop_movement_figure)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 1l. Theta does not predict the outcome or the reaction time, CMR and LOCAL
A: theta rise during the hold (hold − pre; per channel mean over trials, median over channels; one dot = one session) in long trials:
correct, late, early held >= 0.85 s (>= 5 trials per condition and session). Paired sign-flip late − correct and early − correct.
B: within each session (>= 10 correct long trials), Spearman rho between pre-tone theta (pre_tone − pre, median over channels per trial)
and reaction time (release − tone); one dot = one session (colour = rat); sign-flip of rho across sessions.
FDR (Benjamini–Hochberg) over the 3 tests per reference -> q.
Result: CMR late − correct [ ] %, early − correct [ ] %, rho [ ]; LOCAL [ ] %, [ ] %, rho [ ] (earlier runs: rho ≈ 0 in both).
amir_data\Theta\reaction time\theta_outcome_rt_CMR.png, theta_outcome_rt_LOCAL.png

**Why:** if theta during the hold reflected attention / readiness, it should differ before errors or predict the RT.

</div>

In [ ]:
fxr.run(fxp.theta_outcome_rt)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 1m. Theta wait vs go, paper-style (Fakhraei 2021, Fig. 6), CMR and LOCAL
Wait = correct long (tone at 1.05 s), go = correct short (tone at 0.35 s), press-aligned; both identical until 0.35 s.
Theta (4–7 Hz) from the saved wait/go maps (tables\wait_vs_go_maps_<ref>.npz, from the maps cell): mean over 4–7 Hz,
change vs pre-press per session; mean ± SEM across sessions.
As in the paper: t-test across sessions at every time point, FDR over time; a point is marked only if
wait > baseline AND wait > go (both q < 0.05); bar = significant points, q shown = the larger of the two.
Result: CMR wait > go [ ]–[ ] s (earlier run: from about 0.5 s); LOCAL [ ]–[ ] s (earlier run: 0.50–1.19 s).

**Why:** the comparison of the paper (Fakhraei 2021, Fig. 6): wait (long) trials keep the rat waiting after 0.35 s, go (short) trials do not.
Colours: wait = long = green, go = short = red.

</div>

In [ ]:
fxr.run(fxp.theta_wait_vs_go)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 1n. Theta time course: correct vs early and correct vs late long, CMR and LOCAL
Curves from the band-power files (06): per trial and channel change vs own pre-press, only while on the lever;
per session (both groups >= 5 trials) median over channels; time point kept if >= 50 % of the group's trials are on the lever.
Across sessions: mean ± SEM (session = unit). Early long = released before the tone, held > 0.3 s.
Test: paired t-test across sessions (other − correct) at every shown time point, FDR over all shown points of both panels;
black bar = q < 0.05. Window names shown above (no shading).
Result: CMR early vs correct [ ]; late vs correct [ ]; LOCAL [ ]; [ ] (from the printout).

**Why:** shows when (in time) the outcomes differ, without choosing a window first.

</div>

In [ ]:
fxr.run(fxp.theta_course_correct_vs_errors)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 1n-b. Theta: correct vs errors (late + early), everything vs before the press; CMR and LOCAL
**What:** correct long vs error long trials (late and early pooled, and each alone) during the hold and after the release, both relative to before the press.
**Computation:** per trial and channel log10(window / pre); hold − pre uses early trials held ≥ 0.85 s (hold window fully on the lever), post − pre uses
early held > 0.3 s; mean over trials, median over channels → one value per session and group.
**Test:** paired sign-flip across sessions (error − correct); FDR over the tests of one reference.
**Why:** the drop after the release differs by outcome (feedback / confidence idea); this cell gives those numbers with one common baseline.

</div>

In [ ]:
fxr.run(fxp.theta_correct_vs_errors_vs_pre)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 1q-b. Theta after the release: correct vs early, per region, IL vs PL
**What:** is the different theta after the release (correct vs early) present in each region, and is it bigger in IL or PL?
**Computation:** per session and region (median over the region's channels): post − hold for correct long and early long trials; early − correct per session.
**Test:** sign-flip across sessions for each region, and IL − PL (paired, sessions with both).
**Why:** the coupling results are IL–PL; this checks whether the outcome effect after the release is shared by both regions.

</div>

In [ ]:
fxr.run(fxp.theta_after_release_regions)

<div style="color:#FFA000; border-left:4px solid #FFA000; padding-left:10px">

### Delta and theta with the ERP removed (total vs induced)

</div>

<div style="color:#FFA000; border-left:4px solid #FFA000; padding-left:10px">

### Delta vs theta time course around the press and the release (correct long), quick look
**What:** delta (1–3 Hz) and theta (4–7 Hz) change curves over time, side by side.
**Computation:** saved curves of 06 (log10 power / own pre-press power, only while on the lever), median over channels, mean ± SEM over sessions. No test.
**Why:** delta rises around the press and the release (movement-locked); this shows how its time course differs from theta before the
total-vs-induced test. Colours: delta amber, theta green.

</div>

In [ ]:
fxr.run(fxp.delta_theta_quick_look)

<div style="color:#FFA000; border-left:4px solid #FFA000; padding-left:10px">

### 1s. Total vs induced power (mean ERP removed), delta and theta, correct long, LOCAL
**What:** how much of the delta and theta power around the press / release is the evoked potential (ERP, the same wave in every trial)?
**Computation:** per session and channel the mean over trials (ERP) is subtracted from every trial before the Morlet transform → **induced** power;
without subtraction → **total** power. Windows as 1e; evoked share = 1 − induced / total around the release.
**Test:** sign-flip across sessions (total − induced per window; hold vs pre with and without the ERP).
**Why:** delta changes could just be the ERP of the movement; if the delta effect disappears when the ERP is removed, delta is not a sustained
oscillation linked to the action, while theta during the hold stays.

</div>

In [ ]:
fxr.run(fxp.total_vs_induced)

<div style="color:#FFA000; border-left:4px solid #FFA000; padding-left:10px">

### 1s-plot. Theta + delta: total vs induced power, report figure
**What:** report figure of 1s (no recompute; uses the curves of 1s, which is run first if needed).
**Computation / test:** as 1s (solid = total, dashed = induced; press | release). Colours: theta green, delta amber.

</div>

In [ ]:
fxr.run(fxp.total_vs_induced_figure)

<div style="color:#FFA000; border-left:4px solid #FFA000; padding-left:10px">

### 1m-delta. Delta (1–3 Hz) go vs wait from the saved maps (as Fig. 3, mirrored: go > pre AND go > wait), CMR and LOCAL
**What:** is delta higher in go (short) than in wait (long) trials after 0.35 s, i.e. linked to the action?
**Computation:** saved wait / go maps (tables\wait_vs_go_maps_<ref>.npz), mean over 1–3 Hz, change vs pre-press per session; mean ± SEM across sessions.
**Test:** as 1m: t-test across sessions at every time point, FDR over time; a point is marked if go > baseline AND go > wait (both q < 0.05);
sign-flip for the window values.
**Why:** mirror of the theta wait-vs-go test, for delta. Colours: wait = long = green, go = short = red.

</div>

In [ ]:
fxr.run(fxp.delta_go_vs_wait)

<div style="color:#6a1b9a; border-left:4px solid #6a1b9a; padding-left:10px">

### High gamma and multi-unit firing

</div>

<div style="color:#6a1b9a; border-left:4px solid #6a1b9a; padding-left:10px">

### Multi-unit firing (MUA) during the hold vs before the press and after the release (correct long)
**What:** does spiking drop during the hold, as high gamma does?
**Computation:** no spike sorting: threshold crossings on the raw 30 kHz AP band (as is common for MUA). Per session: raw ap.bin (as 02) → same channels as the
1000 Hz files (bad channels out) → band-pass 300–6000 Hz → phase shift → CMR. Noise per channel = MAD / 0.6745 from 20 random 1-s pieces; spike =
downward crossing of −K_THR × noise, 1-ms dead time. Windows as band power: pre −0.5…−0.2 s, hold 0.2–0.85 s (press), post 0.2–0.5 s (release).
Up to N_TR correct long trials per session (random, seed 0). Rate per channel = mean over trials (spikes/s); channels with < 1 spike/s before the press
left out. Session value per region = median over its channels of log10(hold / pre) and log10(hold / post).
**Test:** sign-flip across sessions (fn.sign_flip_p); q = FDR over the 8 tests (4 groups × 2 comparisons). Link to high gamma: per session MUA hold vs pre
against high gamma hold vs pre (CMR, M1 table), Spearman across sessions.
**Why:** high gamma in the LFP follows local spiking; if spiking drops while the rat holds, the high-gamma drop reflects less local firing
(inhibition while waiting). Slow the first time (raw data); one cached file per session in tables\mua_hold\.

</div>

In [ ]:
fxr.run(fxp.mua_hold_vs_pre_post)

<div style="color:#6a1b9a; border-left:4px solid #6a1b9a; padding-left:10px">

### Figure: spiking and high gamma drop during the hold — A one example trial, B the same channel over all its trials, C bars of all sessions
**What:** illustration of the MUA result.
**Computation:** example session = the one whose mPFC MUA drop is closest to the mean of all sessions (≥ 40 trials); channel = an mPFC channel with
≥ 10 spikes/s before the press and a clear drop; trial = the trial whose hold / pre ratio is closest to the channel's mean (typical, chosen by rule).
AP band processed as the MUA cell; high gamma = Morlet 65–96 Hz (10 cycles) of the same channel, amplitude smoothed over 50 ms, relative to the trial's
own pre-press mean. C = the session values of the MUA cell (which is run first if needed). No new test.

</div>

In [ ]:
fxr.run(fxp.mua_gamma_figure)

<div style="color:#6a1b9a; border-left:4px solid #6a1b9a; padding-left:10px">

### High gamma + spikes over time (all sessions) and one real trace with the spikes (correct long, CMR)
**What:** time course of high gamma and of the firing rate around the press and the release, and one raw trace.
**Computation:** A: high gamma = saved 06 curves (log10 power / own pre-press, only while on the lever, median over channels); spikes = cache
tables\mua_curves (made in 06: mua_curves; threshold crossings, 50-ms bins, mPFC channels ≥ 1 spike/s before the press, log10(rate / own pre rate),
median over channels). Mean ± SEM over sessions, % vs before the press. B: one real trial of one channel (spike band 300–6000 Hz, CMR), every detected
spike (crossing of −4.5 × noise) marked, spike count per window on top; typical channel and trial chosen by rule (closest to the median / mean). No test.
**Why:** shows that firing and high gamma fall together during the hold and recover after the release.

</div>

In [ ]:
fxr.run(fxp.gamma_spikes_over_time)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

## 2. Phase coupling between IL and PL (theta)
Are the theta phases of IL and PL locked while the rat holds the lever, in every outcome, and is IL's phase ahead of PL's?
Order: **pilot** (choose pair and reference) → **rat 3** (A1) → **three rats** (A2b, A2c equal n, A2d unbiased) → **controls** (T4 power, T5 matched)
→ **over the trial** (coupling over time, T14 windows, onset timeline, late trials) → **Jäckel et al. 2024 replication** (T8b, Fig. 3D, T8c; δ and θ)
→ overnight summaries. Reference: same-region ring 30–500 µm (SR30-500) unless stated. Inputs made in 06 (coupling data steps):
per-trial phase files, T8c sums, overnight rows (T8, T11, T13–15), T13 fix.

</div>

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 2a. Pilot: choosing the pair and the reference

</div>

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 1o. Theta phase connectivity (PLV, PLI) — step 1: choose the channel pair
Pilot session 20260529_r0000136_wt_1318, the same 20 channels as 1c/1d (fn.select_channels, n_total = 20, from the CMR band-power file).
Per channel: shank (x // 250 µm), x, y (µm; larger y = higher on the probe), region (channel_regions_all.csv), theta hold-specific in 1d
(sign-flip across the 49 correct long trials, hold − pre and hold − post both p < 0.05, same direction; raw p), present in the LOCAL file.
Candidates: lowest and highest channel on each shank, and their distances. The pair for PLV / PLI is chosen from this list.

**Why:** the coupling needs one IL and one PL channel; they are chosen on the pilot session before any coupling is computed.

</div>

In [ ]:
fxr.run(fxpi.pilot_channels_probe)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 1p. Theta phase coupling between three region pairs during the hold: PLV, PLI, PPC, debiased wPLI² — pilot session, correct long trials, CMR
Pilot session 20260529_r0000136_wt_1318, 49 correct long trials (all still holding until 0.85 s), reference CMR. Pairs (20 pilot channels,
theta hold-specific in 1d, different shanks): IL–Cg1 AP49–AP205 (3.7 mm), IL–PL AP49–AP241 (2.2 mm), PL–Cg1 AP257–AP156 (1.8 mm).<br>
Computed with MNE-Python (mne-connectivity 0.9.0, spectral_connectivity_epochs, mode "cwt_morlet", 4, 5, 6, 7 Hz, 5 cycles): across trials at each
time point and frequency, then mean over the 4 frequencies and over the hold window (0.2–0.85 s after the press, 10-ms steps).<br>
- <b>PLV</b> (Lachaux 1999): consistency of the phase difference (counts 0°/180°; biased upward with few trials).
- <b>PPC</b> (Vinck 2010): the same without trial-number bias (≈ PLV²).
- <b>PLI</b> (Stam 2007): how consistently one region is ahead (ignores 0°/180°, i.e. shared signals; biased with few trials).
- <b>Debiased wPLI²</b> (Vinck 2011; MNE "wpli2_debiased"): weighted PLI, squared scale, no trial-number bias, robust to noise.
Mean Δφ: circular mean (> 0 = first region leads).<br>
- Chance: second channel taken from another trial (500 shuffles; own code, identical to MNE within 0.0008 — see validation cell);
black line = 95 % of the shuffles; p = (1 + shuffles ≥ real value) / 501 (raw, one-sided).<br>

**Why:** first look at coupling with several measures; CMR shares all channels between the two signals, so it is replaced by a local reference in 1q.

</div>

In [ ]:
fxr.run(fxpi.pilot_coupling_cmr)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Theta left in LOCAL (% of CMR), hold, correct long: one point per session, regions in colour (representative channels)
**What:** how much theta is removed by the LOCAL reference (channels 30–200 µm away)?
**Computation:** per session and representative channel: theta power in the hold with LOCAL divided by the same with CMR (%). No test.
**Why:** LOCAL removes most of the theta (theta is spatially broad), so it cannot be used for theta coupling; this motivates the same-region ring.

</div>

In [ ]:
fxr.run(fxpi.theta_left_local)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### The most local reference that still has theta: same-region rings, theta peak by FOOOF, pilot, correct long, hold
**What:** which ring (inner–outer distance) of same-region channels can be subtracted while a theta peak remains?
**Computation:** pilot session, original recording (all channels). For 5 seed channels (IL AP49, PL AP241, Cg1 AP205, PL AP257, Cg1 AP156) and every ring
(inner 30–500 µm, outer 150–1000 µm, ≥ 5 channels of the same region): channel − median of the ring; Morlet power 2–30 Hz in the hold; FOOOF fit
(aperiodic fixed, peaks 1–8 Hz wide) → is there a peak with centre 4–8 Hz, its height (log10 above 1/f), and % theta left vs the original.
**Why:** the reference must be local (so IL and PL do not share a subtracted signal) but must not remove the theta itself → the chosen
same-region ring 30–500 µm (SR30-500). Note: the recording, its channel ids, regions and distances were made in an old cell (1q-A) that is gone;
these few lines are now at the top of this cell.

</div>

In [ ]:
fxr.run(fxpi.reference_scan)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 1q. Theta phase coupling of the three pairs with a same-region local reference (30–500 µm) — pilot, correct long, hold
Reference chosen before computing coupling (pilot scan): each channel minus the median of the channels in the SAME region 30–500 µm away
(original recording). Rules for re-referencing (Bastos & Schoffelen 2016; Trongnetrpunya et al. 2016): the two signals share no subtracted channel,
and no channel of the partner's region is subtracted. 30–500 µm is the most local same-region ring with a theta peak above the 1/f background
(FOOOF, centre 4–8 Hz) at all five pilot channels (theta left: IL AP49 22 %, PL AP241 17 % of original). CMR (shares all channels) and
LOCAL 30–200 µm (theta removed) are not used.<br>
Coupling as 1p: MNE-Python spectral_connectivity_epochs (PLV, PLI, PPC, debiased wPLI²), across trials per time point and frequency (4–7 Hz, 5 cycles),
mean over frequencies and the hold window (0.2–0.85 s); chance from 500 trial shuffles (own validated code); raw one-sided p.<br>

**Why:** the same pilot test with the chosen reference: is IL–PL theta coupling still present when no shared signal is subtracted?

</div>

In [ ]:
fxr.run(fxpi.pilot_coupling_sr)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Step A-136: IL–PL theta coupling with the pilot channels in every session of rat 3 (SR30-500, correct long, hold)
**What:** does the pilot pair (IL AP49 – PL AP241) show coupling in the other sessions of rat 3 that have the same channels?
**Computation:** per session: same channels, SR30-500, correct long trials, hold; PLV and PLI over trials, chance from trial shuffles; one row per session
(saved table rat136_pilot_channels_sessions.csv, re-used by the next cell).
**Test:** sign-flip across sessions of (real − chance). Figure: sessions averaged (pilot left out).
**Why:** the pilot was one session; this is the first replication across sessions.

</div>

In [ ]:
fxr.run(fxpi.rat136_sessions)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### IL phase ahead of PL: example trial + phase difference per session (polar) — rat 3, pilot channels, SR30-500, correct long
**What:** in which direction is the phase difference?
**Computation:** example trial chosen by rule (typical phase difference, high amplitude): theta-filtered IL and PL with their peaks; per session the
circular mean Δφ (> 0 = IL phase ahead) as an arrow. No test here (the direction test across sessions is the Rayleigh test of A1).
**Why:** coupling says the phases are locked; the sign of Δφ says which region's phase is ahead (not who drives whom).

</div>

In [ ]:
fxr.run(fxpi.il_ahead_example)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### All pairs: IL–PL theta PLV and PLI, correct long, all sessions — SR30-500, hold, every 4th channel, pairs ≥ 1000 µm, vs chance
**What:** is IL–PL coupling found only for the chosen pair, or for most IL–PL pairs?
**Computation:** theta signal of all channels (06 step 8, SR30-500); every 4th channel; all IL–PL pairs ≥ 1000 µm apart (channel selection: 06 step 11);
per pair and session PLV and PLI in the hold, chance from trial shuffles; per session the mean over pairs. Resumes after every session.
**Test:** across sessions (all, and per rat): sign-flip on (real − chance), raw p. Figure: per rat, bar = mean ± SEM, grey dots = sessions, black line = chance.
**Why:** guards against a lucky pair: the result should hold for the population of IL–PL pairs.

</div>

In [ ]:
fxr.run(fxpi.all_pairs)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 2b. Rat 3 and the three rats (A1–A2d)

</div>

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### A1. Rat 3, correct trials, press → +1 s: pair map, pilot, example trial, sessions, direction
**What:** the coupling story on rat 3 step by step: where the pair is on the probe, the pilot session, one example trial, every session, the direction.
**Computation:** IL (282, 0) – PL (532, 2220) of rat 3 (Reference and window come from `A_RUN` in the settings (main run: 1 = SR30-500, press → +1 s).). Per session: correct long trials (≥ 10); Morlet 4–7 Hz (5 cycles) of IL and PL around the press,
10-ms steps from the press to the window end, each trial only while the paw is on the lever (time points with ≥ 5 trials on the lever).
PLV = |mean of unit phase-difference vectors|, PLI = |mean sign of the imaginary part| (over trials, then averaged over frequencies and time).
Chance: 500 trial shuffles (derangement: PL of another trial). Δφ = circular mean phase difference (> 0 = IL phase ahead).
**Test:** each session vs its own shuffles (p = (1 + shuffles ≥ real) / 501); across sessions one-sided sign-flip of (real − shuffle mean); direction:
Rayleigh test over the session Δφ.
**Why:** one rat, one pair, all steps visible, before pooling rats (A2b). Figures F1–F6.

</div>

In [ ]:
fxr.run(fxco.A1_rat136)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### A2b. Three rats, correct / late / early, press → +1 s (all trials, ≥ 5 per condition)
**What:** is IL–PL theta coupling above chance in each outcome, in all three rats?
**Computation:** one chosen IL–PL pair per rat (PAIRS3; same geometry as the pilot); as A1 for each condition (correct, late, early long; early held ≥ 0.3 s),
sessions with ≥ 5 trials of that condition; per session PLV, PLI, shuffle mean and 95 % of the shuffles, Δφ. Resumes per session (rows file). Reference and window come from `A_RUN` in the settings (main run: 1 = SR30-500, press → +1 s).
**Test:** pooled over the 3 rats with every rat weighted equally (strat_test: sign-flip of real − chance, one-sided).
Figures: pooled bars per condition, circle of correct Δφ per session (Rayleigh), one row per rat.
**Why:** generalises A1 to three rats and all outcomes. Note: PLV and PLI are biased upward with few trials, so outcomes with different trial numbers
are compared in A2c (equal n) and A2d (unbiased measures).

</div>

In [ ]:
fxr.run(fxco.A2b_three_rats)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Phase direction per condition, all trials (from A2b)
**What:** one circle per condition (correct, late, early), one arrow per session in the condition colour, black arrow = mean direction.
**Computation:** session Δφ from A2b (run first if needed). **Test:** Rayleigh test per condition; "IL phase ahead in n / N sessions" (sin Δφ > 0).
**Why:** is IL's phase ahead of PL's in every outcome, not only in correct trials?

</div>

In [ ]:
fxr.run(fxco.direction_circles_all)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### A2c. Three rats, correct / late / early, press → +1 s, the SAME number of trials per condition and session
**What:** A2b with the trial-number bias removed by equal n.
**Computation:** per session and condition: 20 random draws of N_EQ = 10 trials (sessions with ≥ 10 trials of that condition); per draw PLV, PLI and 25
shuffles; session value = mean over draws; chance = mean of the shuffles of the same n. Resumes per session.
**Test:** strat_test (rats equal weight, one-sided) of real − chance per condition.
**Why:** with 10 trials everywhere, a difference between conditions cannot come from different trial numbers. Result (last run): correct PLV / PLI p < .001
(30 sessions); late n.s. (11); early PLI p = .040, PLV n.s. (24).

</div>

In [ ]:
fxr.run(fxco.A2c_equal_n)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Consistency per condition (equal trial numbers, from A2c): direction circles + sessions above chance
**What:** per condition, how many sessions are above their own chance, and do their Δφ point the same way?
**Computation:** A2c rows (run first if needed): session above chance = real > 95 % of its shuffles; circle = one arrow per session, condition colour.
**Test:** Rayleigh test of the session Δφ; count of sessions above chance.
**Why:** a pooled p can be driven by a few sessions; this shows how consistent the effect is across sessions.

</div>

In [ ]:
fxr.run(fxco.consistency_circles_equal_n)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### A2d. Three rats, correct / late / early, press → +1 s, ALL trials, unbiased measures: PPC, debiased PLI², debiased wPLI
**What:** A2b with measures whose expected value does not depend on the number of trials, so all trials can be used.
**Computation:** per session and condition (≥ 5 trials): PPC (Vinck 2010: mean cosine of the phase difference between all pairs of trials),
debiased PLI² and debiased wPLI (Vinck 2011), each trial only while held; chance = the same measure after trial shuffles. Resumes per session.
**Test:** strat_test (rats equal weight, one-sided) of real − chance.
**Why:** keeps every trial (more power than A2c) and still removes the trial-number bias.

</div>

In [ ]:
fxr.run(fxco.A2d_unbiased)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 2c. Controls: power (T4) and matched trials (T5)

</div>

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### T4. Power control: hold vs before / after with theta amplitude matched (PPC + PLI, correct long, SR30-500)
**What:** is the higher coupling during the hold only because theta power is higher during the hold (higher power → cleaner phase)?
**Computation:** ERP removed; windows pre (−1…0 s, press), hold (0…3 s, while held), post (0…1 s, release). Per trial the mean theta amplitude of IL and PL;
trials matched by dropping the strongest hold trials and the weakest pre / post trials until the mean hold amplitude ≤ the other (equal n).
PPC and PLI per time point over the held trials (≥ 5), averaged over time and frequencies.
**Test:** hold − pre and hold − post per session, strat_test (rats equal weight).
**Why:** if coupling is still higher during the hold after matching the amplitude, the coupling change is not a power artefact.

</div>

In [ ]:
fxr.run(fxco.T4_power_control)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### T5. Is there coupling in early / late / errors? Correct matched to the same n and window
**What:** each error condition compared with correct trials of the same session, with the same number of trials and the same window.
**Computation:** window 0–0.5 s after the press, held trials only, ERP removed; correct trials drawn to the n of the error condition (repeated draws);
PLV and PLI vs trial-shuffle chance (derangement). Resumes per session.
**Test:** strat_test (rats equal weight): error vs chance, correct (matched) vs chance, and error − correct.
**Why:** the fairest comparison between outcomes: same window, same n, same sessions.

</div>

In [ ]:
fxr.run(fxco.T5_matched)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### T5 result for the slide: theta coupling during the hold (0–0.5 s, held trials, ERP removed), equal trial numbers
**What:** the T5 numbers as one slide figure: bars = PLV and PLI per condition (3 rats, rats equal weight); dashed line = chance for the same number of
trials; stars = the T5 tests (T5 is run first if needed).

</div>

In [ ]:
fxr.run(fxco.T5_slide)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Slide figure: bias-corrected theta PLV (√PPC) and PLI (√ debiased PLI²) during the hold, correct / early / late, equal trial numbers
**What:** coupling per condition on a scale where chance is 0 for every condition.
**Computation:** per-trial phases (06: trial_phase_files), hold window, equal trial numbers per condition and session; √PPC and √ debiased PLI²
(sign kept) per session. **Test:** strat_test (rats equal weight) vs 0.
**Why:** removes the trial-number bias and makes the conditions directly comparable on one axis.

</div>

In [ ]:
fxr.run(fxco.bias_corrected_slide)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 2d. Over the trial: when are IL and PL coupled?

</div>

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Theta coupling over the trial, correct / early / late: around the press (−0.5 → 1.0 s) and the release (−0.5 → +0.5 s)
**What:** when during the trial is IL–PL theta coupled?
**Computation:** measure = bias-corrected PLI (√ debiased PLI²; chance = 0 for every condition). Press panel: saved trial phases (06), trials held ≥ 0.5 s up to
0.5 s (as T5), then held to the end of each step. Release panel: computed once and saved (resumes). Mean over rats (rats equal weight).
**Test:** filled marker = above chance in that time step (strat_test, p < 0.05, raw).
**Why:** shows that coupling builds up during the hold and is lost after the release.

</div>

In [ ]:
fxr.run(fxco.coupling_over_trial)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Each condition vs its own chance in each 0.2 s window after the press (T14, equal trial numbers, ERP kept)
**What:** per 0.2-s window after the press, is each condition coupled?
**Computation:** T14 rows of the overnight run (06): equal trial numbers per condition pair, ERP kept; per session real − chance per window.
**Test:** strat_test (rats equal weight) per condition and window, raw p.
**Why:** the time-resolved version of T5 with equal n.

</div>

In [ ]:
fxr.run(fxco.T14_vs_chance)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### When is each condition coupled? Timeline of windows significantly above chance (T14)
**What:** one line per condition; a dot where the window is above chance (from the previous cell's test), first dot at the window start.
**Computation / test:** as the previous cell (T14 rows, strat_test, p < 0.05). Time (s) from the press; tone of long trials at 1.05 s.
**Why:** a compact picture of the onset of coupling in correct, late and early trials.

</div>

In [ ]:
fxr.run(fxco.onset_timeline)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Consistency across sessions of each onset point: sessions above chance (T14) + direction circles
**What:** for each onset window: how many sessions are above chance, and does Δφ point the same way?
**Computation:** T14 rows (sessions above their own chance) and saved trial phases (06) for the session Δφ in that window.
**Test:** Rayleigh test of the session Δφ; count of sessions above chance.
**Why:** checks that each "significant" window is carried by many sessions with the same direction.

</div>

In [ ]:
fxr.run(fxco.onset_consistency)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Late trials: theta coupling across the hold (T14 windows, vs chance) + direction circle for 0.6 s → tone
**What:** late trials (slow responses) analysed alone across the whole hold.
**Computation:** 1. T14 rows, pair correct-late, B = late, equal n: real − chance per 0.2-s window. 2. saved trial phases: late trials, 0.6 s → tone,
one arrow per session (circular mean Δφ).
**Test:** strat_test per window (rats equal weight); Rayleigh test for the circle.
**Why:** late trials had no overall coupling in A2c; this asks whether they couple late in the hold (just before the tone).

</div>

In [ ]:
fxr.run(fxco.late_across_hold)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Late trials, 0.8–1.0 s after the press (just before the tone): consistency across sessions + direction circle
**What / computation:** as the previous cell, only the last window before the tone: sessions above chance and one arrow per session.
**Test:** Rayleigh test; count of sessions above chance.

</div>

In [ ]:
fxr.run(fxco.late_before_tone)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### 2e. Replication of Jäckel et al. 2024 (delta and theta) and the overnight tests

</div>

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### T8b. Jäckel et al. 2024 replication done their way (sessions pooled per rat) + within-rat shuffle + session PPC (theta and delta)
**What:** the phase coupling analysis of Jäckel et al. (2024): PLV between PL and IL at the press and the release, correct vs early.
**Computation:** FIR band-pass (delta 2–4 Hz, theta 4–8 Hz, filters as in the paper) + Hilbert phase; phase difference of every trial at t = 0
(ERP kept); PLV with the trials of all sessions of one rat pooled (as the paper); chance = within-rat shuffle; also PPC per session (unbiased). Resumes.
**Test:** pooled PLV vs within-rat shuffle; session PPC with strat_test.
**Why:** to see whether our data reproduce the published result when analysed exactly the published way, and how that changes with unbiased per-session statistics.

</div>

In [ ]:
fxr.run(fxco.T8b_jaeckel)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Replication of Jäckel et al. 2024 Fig. 3D: PLV at press / release per rat, δ and θ (from T8b_trials.csv)
**What:** the paper's Fig. 3D with our rats: PLV at the press and at the release, correct vs early, one value per rat (sessions pooled). No new test.

</div>

In [ ]:
fxr.run(fxco.T8b_fig3D)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### T8c. Jäckel Fig. 3C replication, ERP removed — corrected jackknife
**What:** PL–IL coupling over time around the press and the release (δ and θ), correct vs early, as Fig. 3C of the paper but with the ERP removed.
**Computation:** per session the sums of the unit phase-difference vectors (06: T8c_sums; ERP = condition mean subtracted per session before filtering);
pooled per rat → PPC (not biased by trial number); curves = √PPC (sign kept).
**Test:** jackknife over sessions → t with (sessions − 1) df → z with the same tail probability → Stouffer over the 3 rats. Coloured bars = condition coupled
(PPC > 0, one-sided); black bars = early vs correct differ (delta, two-sided); runs shorter than 50 ms are not drawn.
**Why:** the paper pooled trials and kept the ERP; with the ERP removed and an unbiased measure the remaining effect is the real phase coupling.

</div>

In [ ]:
fxr.run(fxco.T8c_fig3C)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Summaries of the overnight tests (made in 06): T8 θ / δ, T11, T13–15, T13 fix
**What:** prints the summary files written by the 06 data steps, and shows the T8 trace figures.
- **T8** (theta 4–8 Hz, delta 2–4 Hz): Jäckel replication per session (FIR + Hilbert, PLV / PLI around press and release, correct vs early; draws of equal n, shuffles).
- **T11**: is coupling explained by theta amplitude / power? Per session, coupling of high- vs low-amplitude trials.
- **T13–15**: coupling in windows (T13 before the tone, T14 0.2-s windows after the press, T15 around the release), equal trial numbers, ERP kept;
  **T13 fix**: the tone column fixed, −0.5 → 0 s before the tone, correct vs late.
Tests are inside the summaries (strat_test, rats equal weight; two-sided where a difference is tested).

</div>

In [ ]:
fxr.run(fxco.overnight_summaries)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

## 3. Beta bursts (15–30 Hz) and gamma bursts as control
Bursts as Karvat et al. 2021 (06 steps 2–4): Morlet wavelets (7 cycles beta, 10 gamma), a burst = power above the 90th percentile of that frequency
over the session; occupancy = % of time inside a burst; LOCAL reference; one representative channel per region (IL, PL, Cg1; pooled = their mean).
Steps 1–2: more bursts before late responses? · Step 5: occupancy and success · Step 6: last burst and RT · Step 7: after an error ·
Steps 8–10: what the bursts look like (98th percentile, Karvat 2020). Gamma bursts (30–100 Hz) are the control band (see the gamma box below).
The along-the-session steps (0, 3, 4, Gamma Step 1) are in section 4.

</div>

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Burst tables — question 1: more or fewer beta bursts before a late response?
Burst occupancy (Karvat 2021 rule: Morlet 7 cycles, 90th percentile per frequency over the whole session; one representative channel per region;
pooled = mean of PL, IL, Cg1 per trial), LOCAL reference.
Main test: late long vs correct long, occupancy in the 0.6 s before the tone (beta 15–30 Hz; gamma 30–100 Hz as comparison).
Session value = 100 × (mean occupancy late − mean occupancy correct), %-points; POSITIVE = more bursts before the late response (paper direction).
Across sessions: sign-flip (10,000 flips) and paired t-test; FINDING = both p < 0.05. Checks: leave-one-session-out (largest p),
without channels > 300 µm from the representative position, per rat. Raw p, no FDR.
Filter: all blocks, no minimum correct RT, sessions need ≥ 1 trial of each kind (MIN_TR = 1).

**This cell** loads the burst trial tables of all sessions (06 steps 2–3) for beta 15–30 Hz and gamma 30–100 Hz: per trial, region and window the
burst occupancy (% of time in a burst) and count. Used by every burst cell (they run it first if needed).

</div>

In [ ]:
fxr.run(fxb.load_burst_tables)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### The selected channel per region (beta bursts) in brain coordinates (coronal), supplementary figure
**What:** where the representative IL, PL and Cg1 channels lie in the brain, all sessions. **Computation:** atlas coordinates of the representative channels
(01, 06), coloured by region. No test. **Why:** shows that "IL" and "PL" channels are in the same place across sessions and rats.

</div>

In [ ]:
fxr.run(fxb.burst_channels_brain)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

<b>Step 1-power: beta power around the tone, correct vs late long trials, as Karvat et al. (2021), Fig. 2D</b><br>
Beta 15–30 Hz (Morlet, 7 cycles), each frequency normalised to the 90th percentile of its artifact-free power, representative channel (LOCAL).
−0.5 to +0.5 s around the tone in 25-ms bins. Two-way repeated-measures ANOVA (time × outcome), trials as units, Tukey post hoc per bin;
grey: p &lt; 0.01. Mean ± 95 % CI over trials.

**Why:** bursts are a threshold on power; this shows the same comparison on the power itself, as Karvat et al. (2021) did.

</div>

In [ ]:
fxr.run(fxb.beta_power_karvat)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Beta power, pooled mPFC: correct vs late long trials around the tone (paper style; shown only)
**What / computation:** as Step 1-power, pooled over IL, PL and Cg1 (uses the bins of the previous cell, run first if needed). No new test.

</div>

In [ ]:
fxr.run(fxb.beta_power_pooled)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

<b>Step 0: late − correct beta burst occupancy per region, simple test (session means)</b><br>
Per session: mean occupancy in the 0.6 s before the tone of late minus correct long trials, sessions with ≥ 5 late and ≥ 3 correct trials.
Session values tested against zero with a sign-flip test (10,000 permutations). Bars: mean ± 95 % CI over sessions; dots: sessions.

**Why:** the simplest test (one value per session) before the mixed model of Step 1.

</div>

In [ ]:
fxr.run(fxb.step0_simple)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Step 1. Which region shows more beta bursts before late responses?

Beta bursts (15–30 Hz, Karvat/Golan method), LOCAL reference, one representative channel per region (IL, PL, Cg1) and the mean of the three (mPFC pooled).<br>
Long trials, correct vs late, all blocks, 60 sessions. Value per trial: burst occupancy (% of time in a burst) in the 0.6 s before the tone.<br>
Model per region: occupancy ~ late + position in session + block + (1 | session); the bar is late − correct (%-points) with 95 % CI,
yellow if p < 0.05 (raw p, no FDR).

**This cell (rebuilt):** the old Step 1 computing cell is missing from the notebook. It is rebuilt from the Step 1 description and the Step 3 code:
one row per trial and region (occupancy in the 0.6 s before the tone, LOCAL, long correct / late; position = trial number in the session, as Step 3),
then the model per region. It also prints the table saved by the old Step 1 (Beta bursts\step1_late_vs_correct_regions.csv) — **the two should be the same; if not, tell me.**
It makes D, TP and R1 for the next four cells.

</div>

In [ ]:
fxr.run(fxb.step1_model)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Step 1 figure (report Fig. 8): late − correct beta burst occupancy per region
**What / computation:** the model estimates of Step 1 (late − correct, %-points, 95 % CI) for mPFC pooled, IL, PL, Cg1; blue bar if p < 0.05 (raw).
FDR over the 4 groups is printed as well. **Test:** from the mixed model of Step 1.

</div>

In [ ]:
fxr.run(fxb.step1_regions)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Step 1-check. Simple session test vs mixed model (no block), pooled and each region
**What:** does the Step 1 result depend on the method?
**Computation:** (1) mixed model without block: occupancy ~ late + position + (1 | session); (2) per session mean(late) − mean(correct), sign-flip across sessions,
for ≥ 1, 3, 5 late trials per session (≥ 3 correct). **Why:** robustness of Step 1 to the model and to the minimum number of late trials.

</div>

In [ ]:
fxr.run(fxb.step1_check)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

<b>Step 1b: IL vs PL in the same sessions and trials (beta occupancy, 0.6 s before the tone, LOCAL, long trials)</b><br>
Sessions with both an IL and a PL channel. (1) Per session: late − correct in IL and in PL (≥ 5 late, ≥ 3 correct), and IL − PL of these effects, sign-flip test.
(2) Per trial: IL − PL occupancy, mixed model diff ~ late + position + (1 | session).

**Why:** a region difference must be tested directly (IL − PL), not by comparing two p values.

</div>

In [ ]:
fxr.run(fxb.step1b_il_vs_pl)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

<b>Step 1c: does the IL / PL result depend on the channel? (beta occupancy 0.6 s before the tone, LOCAL, long trials)</b><br>
From the whole-probe burst files (one channel about every 90 µm per shank, same detection): for IL and PL, the representative channel, the mean of all channels
of the region, and the dorsal-most and ventral-most channel. Each version: mixed model occ_pct ~ late + position + (1 | session) and simple test
(session means, ≥ 5 late, ≥ 3 correct, sign-flip).

**Why:** shows that the IL / PL result does not depend on which channel represents the region.

</div>

In [ ]:
fxr.run(fxb.step1c_other_channels)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Step 2. What a beta burst looks like (example in the 600 ms before the tone)
Test session, IL, LOCAL. Among long correct/late trials (blocks 0.5 and ≥ 0.75, no artifacts), the clearest detected beta burst
(15–30 Hz, 50–300 ms) lying fully inside the 600 ms before the tone (largest beta amplitude vs the rest of the window; chosen for illustration).
Rows: raw LFP · beta-filtered signal with its amplitude (envelope) · time–frequency power (Morlet, 7 cycles) / session 90th percentile, 15–40 Hz.
Red = the detected burst (Karvat et al. 2021 method); white outline = above the 90th percentile inside the burst and 15–30 Hz.

**Why:** shows what a detected burst is in the raw signal (illustration only, chosen as the clearest).

</div>

In [ ]:
fxr.run(fxb.step2_burst_example)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Step 5. Do more beta bursts before the tone mean less success?

Same trials as Steps 3–4 (IL, LOCAL, long correct vs late, all blocks, whole session).<br>
Each trial is put in a group by its beta burst occupancy in the 0.6 s before the tone (as Golan / Karvat 2021):
quiet 0 %, low ≤ 25 % (≤ 150 ms), medium ≤ 50 % (≤ 300 ms), high > 50 % (> 300 ms).<br>
Success = correct / (correct + late). For each session and group: success as % change from the session mean
(groups with ≥ 3 trials; groups shown if ≥ 5 sessions; sessions with at least one correct and one late trial).<br>
Tests: one-way ANOVA across groups, and a straight line through the session values (trend = change in success per group step,
same units as the bars; SE clustered by session).

**Why:** links bursts to behaviour on the trial level: fewer successes when more of the pre-tone time is in a beta burst.

</div>

In [ ]:
fxr.run(fxb.step5_success)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Step 5 figure (report Fig. 13 B): success per occupancy group, beta blue from light (quiet) to dark (high)
**What / computation:** the session values of Step 5 (run first if needed), mean ± SEM per group, ANOVA and trend p from Step 5. No new test.

</div>

In [ ]:
fxr.run(fxb.step5_figure)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Step 5b. Example trials for the four occupancy groups (quiet, low, medium, high), IL, LOCAL
**What:** one example trial per group: LFP and 15–30 Hz signal in the 0.6 s before the tone, detected bursts shaded. Chosen as typical (illustration). No test.

</div>

In [ ]:
fxr.run(fxb.step5b_examples)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Fig. 13 (report): A example trials of the four occupancy groups, B success rate per group
**What / computation:** A from Step 5b, B from Step 5 (both run first if needed). No new test.

</div>

In [ ]:
fxr.run(fxb.fig13)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Step 6. Does the timing of the last beta burst before the tone predict the reaction time?

IL, LOCAL, long trials (correct and late), all blocks, exact burst times.<br>
For each trial: the last beta burst that started in the 1 s before the tone and how long before the tone it ended (gap, s).
Trials whose last burst is still ON at the tone are left out (no gap), as are trials without a burst in the 1 s and trials whose
window overlaps an artifact.<br>
<b>Why a simple correlation is not enough</b> (figure step6_RT_why): <b>A</b> the rats differ in RT (mean of session medians ± SEM);
<b>B</b> RT and gap change along the session; <b>C</b> more bursts in the 1 s → the last burst ends closer to the tone.<br>
So the model corrects for all three: log10 RT ~ gap + rat + position in session (z) + number of bursts (z) + (1 | session).
Effect = % change in RT when the last burst ended 1 s closer to the tone.<br>
<b>Figure step6_RT</b>: model line and adjusted RT in 4 ranges of the gap (display only; the test is on the continuous gap).
Checks (+ late term, correct only, late only) are printed and saved for the robustness slide.

**Why:** asks whether the timing of a beta burst (not only how much beta) relates to how fast the rat responds.
(The older Step 6 version with a block term and the IL-vs-PL check 6b were removed.)

</div>

In [ ]:
fxr.run(fxb.step6_rt)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Step 7. After an error, is there more beta in the next trial — before or after the press?

Behaviour: after a late trial the next correct response is ~35 ms slower. Here: correct long trials, IL, LOCAL, all blocks, grouped by the
outcome of the trial directly before: <b>after correct</b> vs <b>after an error</b> (late or early).<br>
Plot: beta burst occupancy from 1 s before to 1 s after the press (mean over trials ± SEM, 150-ms smoothing for display).<br>
Test: <b>sign test across sessions</b> — for each session (≥ 3 trials of each kind), is the mean occupancy higher after an error than after
a correct? Done for the 1 s before and the 1 s after the press. Black = p < 0.05, grey = n.s.<br>
Result: before the press beta is slightly higher after an error (~+1.2 %-points, ~5 %); after the press there is no difference.
Late and early errors separately go in the same direction but are not significant on their own (fewer trials).<br>
Checks: licking does not explain it (licks in 0.2 % of trials in that second); the second before the press never contains the previous trial
(the rat waits ~7–9 s). Exploratory: the 1 s windows were chosen after looking at the time course (cluster test over the whole time course p = 0.077). Raw p.

**Why:** post-error slowing (03) suggests a changed state after an error; beta before the next press is a candidate.

</div>

In [ ]:
fxr.run(fxb.step7_after_error)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### What the bursts look like (Karvat 2020)

</div>

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Step 8. What do the IL beta bursts look like? (as Karvat et al. 2020, Fig. 2; descriptive)

All IL beta bursts (LOCAL, 15–30 Hz; Morlet 7 cycles, peaks in time–frequency above the <b>98th percentile</b> of each frequency, as Karvat 2020)
within the task period (first press → last release) of all 60 sessions, 3 rats pooled.
Each burst (merged time–frequency event) is represented by its strongest peak:
<b>a</b> peak frequency, <b>b</b> frequency span (number of 1-Hz frequencies above threshold at the peak time),
<b>c</b> duration (time above threshold at the peak frequency), <b>d</b> inter-burst interval (strongest peak of one burst → strongest peak of the next),
<b>e</b> power at the peak (FOM = fraction of the median power of that frequency over the session).<br>
y-axis = bursts per minute of task time; dashed line = median. Descriptive only, no statistical tests.<br>
Result: 103,977 bursts (~1,800 per session, ~53 per minute); median 23 Hz, span 4 Hz, duration 67 ms, interval 0.76 s, power 7.4 FOM
(Karvat 2020, rat M1: median duration 70 ms). 1.5 % of bursts last < 10 ms and 2 % of intervals are < 0.1 s
(weak single-frequency events just above threshold).

**Why:** describes our bursts with the same measures as Karvat et al. (2020) so they can be compared.

</div>

In [ ]:
fxr.run(fxb.step8_properties)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Step 9. Single IL beta bursts and their average (as Karvat et al. 2020, Fig. 3a; descriptive)

IL channel, LOCAL reference, raw LFP (µV, 1000 Hz), <b>98th percentile</b> (data from 06 cell 5e).
Bursts as in Step 8 (strongest peak of each merged burst, task period), selected as Karvat 2020 selected his rewarded bursts:
<b>duration ≥ 70 ms</b> and <b>peak frequency 20–25 Hz</b> (70 ms = our median burst duration; bursts are spread evenly over 15–30 Hz, so the middle band was used).
Each burst aligned to its beta trough (15–30 Hz filter, trough nearest the peak); bursts touching an artifact excluded.<br>
<b>Top:</b> 10 random selected bursts per session; black = LFP, colour = time inside the burst (power above the 98th percentile).
<b>Bottom:</b> mean LFP of all selected bursts of the session. Dashed line = trough (0 ms). Same scale bars in all panels.<br>
<b>Sessions:</b> 4 typical sessions (rule fixed before looking: sessions with < 150 selected bursts and 4 unusual sessions excluded; in each rat the session
closest to the rat's median trough depth and clarity, plus the next closest overall), shown in date order as Session 1–4.<br>
Result: single bursts are short and differ from each other, while their average looks like a rhythm (~22 Hz) — ~2.5 cycles over all 60 sessions
(Karvat 2020: ~6 cycles; his raw LFP ~10× larger, see note on reference). Descriptive only, no statistical tests.

</div>

In [ ]:
fxr.run(fxb.step9_single_bursts)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Step 10. Is the average burst shape the same in every session and rat? (descriptive)

Mean LFP of the selected bursts (≥ 70 ms, 20–25 Hz; 98th percentile, IL, LOCAL; as Step 9) in each of the 60 sessions (06 cell 5e),
baseline (|t| > 200 ms) subtracted and divided by the depth of its own trough (trough = −1), so that only the shape is compared, not the size.
Thin lines = sessions (colour = rat), black = median over sessions.<br>
Shape similarity: correlation (±150 ms) of each session's normalized mean with the median of all other sessions.
Rhythm: frequency from the time between the peak before and the peak after the trough.<br>
Note: the size of the bursts differs between rats and depends on the reference (see slide note); here only the shape is compared.
Descriptive only, no statistical tests.

</div>

In [ ]:
fxr.run(fxb.step10_shape)

<div style="color:#6a1b9a; border-left:4px solid #6a1b9a; padding-left:10px">

### Gamma bursts (30–100 Hz): is the beta story specific to beta?

Same burst detection as beta (Karvat 2021: Morlet wavelets, 10 cycles for gamma; peaks above the 90th percentile per frequency; 50-ms bins),
same IL channel, LOCAL reference, same trials and models as the beta Steps. Gamma occupancy is ~47–49 % (beta ~26–28 %).<br>
Quick check of the beta Steps in gamma (IL, LOCAL, raw p):
<ul>
<li><b>Only in beta:</b> more bursts before late responses (Step 1: beta +2.7 %-points, p = 0.005; gamma +0.5, p = 0.56, and n.s. in PL, Cg1, pooled);
the start-of-session effect 0.6 s before the tone (Step 3: beta +8.2, p < 0.001; gamma +2.8, p = 0.10);
fewer successes with more bursts before the tone (Step 5: beta trend −3.2 % per group, p = 0.012; gamma +1.5, p = 0.42).</li>
<li><b>Also in gamma (general state, not beta-specific):</b> bursts decrease over the session (Step 0: gamma decreasing in 49 of 60 sessions in IL, p < 0.001, and in all regions);
more bursts before the next press after an error (Step 7: gamma 37 of 56 sessions, p = 0.022; beta 40 of 56, p = 0.002).</li>
<li>Not checked in gamma: Step 6 (needs exact burst times, beta only) and Steps 8–10 (need a gamma detection at the 98th percentile).</li>
</ul>

</div>

## 4. Along the session (performance, satiety, time)
**How the session is split.** Each trial gets a position from 0 (start) to 1 (end) in three ways: **trial number** (rank of the trial in the session),
**clock time** (press time between the first and the last press) and **water drunk** (share of the session's reward volume received before the trial).
Parts: **start** < 0.25 · **middle** 0.25–0.85 · **end** ≥ 0.85 (the same cuts for all three). For curves the session is cut into 10 (or 20) equal parts.
Using three definitions checks that "start" does not depend on how it is measured (the rats drink, get sated and slower as the session goes on).

Contents: theta along the session (power, three splits) · RT vs theta drop · coupling along the session · beta bursts along the session
(Step 0: no trial categories; Step 3: late vs correct; Step 4: where in the trial) · Gamma Step 1 (beta vs gamma at the session start).
Colours of the boxes = band of each cell.

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Theta along the session

</div>

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Fig. S13. Theta across the session (pooled mPFC, CMR): before the press (dashed) and during the hold (solid)
**What:** does theta change from the start to the end of the session?
**Computation:** correct long trials; per trial log10 theta power before the press and during the hold; session split into parts by position;
per part median over channels, mean over trials → mean ± SEM over sessions.
**Test:** hold vs the session's pre level in each part; start vs middle and end vs middle (paired sign-flip across sessions); FDR over the 5 tests.
**Why:** the rats drink and become sated along the session; a state change would show up here.

</div>

In [ ]:
fxr.run(fxp.theta_across_session)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Theta and high gamma during the hold along the session (correct long, all sessions, CMR)
**What:** the hold-vs-pre change of theta and of high gamma along the session.
**Computation:** value = hold (0.2–0.85 s) vs before the press (−0.5…−0.2 s) of the same trial, log10; per channel mean over trials, median over channels (as 1e).
Position = press time (clock time). A: 10 parts, mean ± SEM over sessions (part drawn if ≥ 3 trials). B: start 0–25 %, middle 25–85 %, end 85–100 %.
**Test:** paired sign-flip across sessions (start vs middle, start vs end; sessions with ≥ 3 trials in both parts); FDR over the 4 tests (2 bands × 2).
**Why:** do the hold effects (theta up, high gamma down) weaken as the session goes on? Colours: theta green, high gamma purple.

</div>

In [ ]:
fxr.run(fxp.theta_gamma_along_session)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Theta during the hold along the session: A by trial number, B start / middle / end with three ways to split
**What:** the same theta value as above, with the three session splits used for the bursts (Step 3).
**Computation:** trial number = rank of the trial in the session; clock time = press time (first → last press); water drunk = share of the session's reward
volume received before the trial. Parts: start < 0.25, middle 0.25–0.85, end ≥ 0.85. A: 10 parts by trial number; B: bars = mean ± SEM over sessions.
**Test:** trial number only: middle or end vs start (paired sign-flip, sessions with ≥ 3 trials in both); FDR over the 2 tests.
**Why:** checks that "start of the session" means the same with each definition (time, trials, water).

</div>

In [ ]:
fxr.run(fxp.theta_three_splits)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Reaction time vs theta drop after the release: long correct + late trials pooled, same number of each, RT 0.1–1.5 s
**What:** are faster responses followed by a bigger theta drop after the release?
**Computation:** CMR band-power files (06). Long trials, RT = release − tone, only RT 0.1–1.5 s. Per session the same number of correct and late trials
(random draw, n = the smaller count, ≥ 5 each). Theta drop per trial = log10(hold / post) (4–7 Hz; hold 0.2–0.85 s after the press, post 0.2–0.5 s
after the release; > 0 = drop), median over channels. Per session the Spearman ρ between drop and RT.
**Test:** sign-flip across sessions of ρ (10,000 flips; mean ρ vs 0); number of sessions with ρ < 0.
**Why:** pooling correct and late gives a wide range of RTs in every session; equal numbers stop one outcome from dominating the correlation.
**Note:** this cell is rewritten from the description of the last version (the old cell is not in the recovered files). It should give
ρ ≈ −0.11, p ≈ .018, 20 of 30 sessions negative. If the numbers differ, tell me (the drop definition — hold − post — is the first thing to check).

</div>

In [ ]:
fxr.run(fxp.rt_vs_theta_drop)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Coupling along the session

</div>

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Theta coupling during the hold along the session (correct long, SR30-500, chosen pairs): A by trial number, B start / middle / end, three splits
**What:** does IL–PL coupling change along the session?
**Computation:** per-trial phases (06: trial_phase_files). Value = PLI − chance (hold 0.2–0.85 s) of the trials in each part; chance = the trials of the same
part shuffled (100 shuffles). Positions and cuts as in the theta-power cell. A: 10 parts by trial number (part used if ≥ 5 trials).
B: start / middle / end, equal n across the parts of a session (20 draws).
**Test:** middle or end vs start, sign-flip across sessions; FDR over the 2 tests of each split.
**Why:** tests whether the coupling result belongs to a particular part of the session (e.g. when the rat is thirsty).

</div>

In [ ]:
fxr.run(fxco.coupling_along_session)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Raw PLV and PLI (no chance) along the session: correct long, hold 0.2–0.85 s, SR30-500; equal n across the parts of each session
**What / computation:** as above, but the raw values (no chance subtracted), equal n per part within each session; mean ± SEM over sessions. No test.
**Why:** with equal n, the raw values can be compared between parts.

</div>

In [ ]:
fxr.run(fxco.raw_plv_pli_along_session)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Raw PLI along the water drunk (5 parts, zoomed, no tests): correct long, hold 0.2–0.85 s, SR30-500
**What / computation:** raw PLI per part of the water drunk (share of the session's reward volume before the trial), equal n per part within each session. No test.

</div>

In [ ]:
fxr.run(fxco.pli_along_water)

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

### Raw PLI along clock time and trial position (5 parts, zoomed, no tests): correct long, hold 0.2–0.85 s, SR30-500
**What / computation:** as the previous cell, with clock time and trial position as the split. No test.

</div>

In [ ]:
fxr.run(fxco.pli_along_clock_trial)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Beta bursts along the session

</div>

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Step 0. How do mPFC beta bursts behave across the session?

All beta bursts (15–30 Hz, LOCAL, exact burst times) over the whole task period of each session, from the first press to the last release,
<b>without any trial categories</b>. Each session is divided into 20 equal parts (0 = start, 1 = end); artifact time is removed
(parts with > 50 % artifact left out).<br>
Rows: <b>mPFC pooled</b> (mean of IL, PL and Cg1, using the regions each session has), then <b>IL</b>, <b>PL</b>, <b>Cg1</b>.
Left: burst occupancy (% of time in a beta burst); right: burst rate (bursts per second). Black = mean over sessions ± SEM, pink = rats.<br>
Test: for each session, Spearman correlation between position in the session and the beta measure; sign test across sessions
(do more sessions go down than expected by chance?). Black text = p < 0.05, grey = n.s. Raw p.<br>

**Why:** before comparing late and correct along the session, how beta itself changes along the session (no trial categories).

</div>

In [ ]:
fxr.run(fxb.step0_across_session)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Beta bursts across the session: figure pooled mPFC (paper style) + numbers for pooled, IL, PL, Cg1
**What / computation:** as Step 0; per-session Spearman (position vs measure); start / end = first / last 10 % of the session.
**Test:** sign test across sessions (binomial); FDR over the 4 groups.

</div>

In [ ]:
fxr.run(fxb.step0_figure)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Step 3. The late − correct difference appears only at the start of the session

IL beta bursts, LOCAL, long trials (correct vs late), all blocks, 60 sessions.<br>
Value per trial: beta burst occupancy (% of time in a burst) in the 0.6 s before the tone; burst rate (bursts/s) is shown in the robustness section.<br>
<b>A</b> occupancy along the session (10 bins of trial position) for correct and late trials, with the straight-line model
occupancy ~ late × position + block + (1 | session).<br>
<b>B</b> late − correct per session phase (start 0–25 %, middle 25–85 %, end 85–100 % of trials), model
occupancy ~ late × phase + block + (1 | session); bars yellow if p < 0.05, dots = rats.<br>
Three ways to split the session (trial number, clock time, water drunk) check that the start effect does not depend on how "start" is defined.

**Why:** the late − correct difference of Step 1 appears only at the start of the session; this tests that with three definitions of "start".

</div>

In [ ]:
fxr.run(fxb.step3_along_session)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Fig. 10. Beta burst occupancy 0.6 s before the tone along the session: correct vs late long trials, 10 equal parts
**What / computation:** mean ± SEM over sessions per part (Step 3 data, run first if needed), straight lines from the mixed model, start / middle / end as in Fig. 9. No new test.

</div>

In [ ]:
fxr.run(fxb.fig10)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### All numbers for late vs correct beta bursts 0.6 s before the tone (LOCAL, no block), IL / PL / Cg1 / pooled
**Computation / tests:** 1. simple test (session means, sign-flip) for the whole session and per part; 2. mixed model per part × 3 splits, all rats and each rat;
3. the same for the burst rate; FDR (Benjamini–Hochberg) over the 4 groups within each test; 4. occupancy levels per part.
**Why:** the full table behind Figs. 10–11 (for the robustness slide).

</div>

In [ ]:
fxr.run(fxb.step3_numbers)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Fig. 11 (all rats) and Fig. S16 (each rat): late − correct beta occupancy 0.6 s before the tone per part of the session
**What / computation:** mixed model per part, no block, 95 % CI; three ways to split the session (Step 3 data). No new test.

</div>

In [ ]:
fxr.run(fxb.fig11)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Missing numbers: burst rate for the whole session (mixed model, 4 groups) and the simple test per part for all three splits (all rats and each rat)
**Computation / tests:** whole session: y ~ late + position + (1 | session), 4 groups, raw p; per part: session means, sign-flip across sessions.

</div>

In [ ]:
fxr.run(fxb.step3_missing_numbers)

<div style="color:#1565c0; border-left:4px solid #1565c0; padding-left:10px">

### Step 4. Where in the trial is the start effect?

Same trials as Step 3 (IL, LOCAL, long correct vs late, all blocks).<br>
Three windows per trial: <b>before press</b> (−1.5–0 s), <b>whole hold</b> (press → tone), <b>0.6 s before tone</b>.<br>
<b>A</b> beta burst occupancy over time in the start phase (first 25 % of trials), correct vs late, from 1.5 s before the press to the tone
(median tone time; dashed lines at press, 0.6 s before the tone and tone).<br>
<b>B</b> late − correct in the start phase for each window, model occupancy ~ late × phase + block + (1 | session).

**Why:** is the extra beta before late responses at the session start already there before the press (a state), or only before the tone?

</div>

In [ ]:
fxr.run(fxb.step4_where_in_trial)

<div style="color:#6a1b9a; border-left:4px solid #6a1b9a; padding-left:10px">

### Gamma Step 1. At the start of the session, is the extra activity before late responses beta-specific? (beta vs gamma)

IL, LOCAL, long trials (correct and late), all blocks, 60 sessions. Start of the session = first 25 % of trials (trial number).
Same model in both panels: value ~ late × phase + block + (1 | session); shown is late − correct at the start ± 95 % CI; filled = p < 0.05, hollow = n.s. (raw p).<br>
<b>A Bursts</b> (50-ms bins; as beta Step 4): burst occupancy before the press (−1.5 to 0 s), during the whole hold (press → tone), in the 0.6 s before the tone.
Shown relative to each band's own level (late − correct as % of the mean correct occupancy of that band at the start), because gamma occupancy (~50 %)
is about twice beta occupancy (~27 %); p values unchanged by this scaling.<br>
<b>B Band power</b> (no burst detection, so not affected by gamma covering more frequencies than beta): log10 power averaged over the band
(beta 13–29, low gamma 30–60, high gamma 61–100 Hz) before the press (−0.5 to −0.2 s), during the hold (0.2–0.85 s), in the 0.3 s before the tone;
late − correct as % change in power.<br>
Result: in both measures, all bands are raised before late trials during the hold (a general state at the session start), but just before the tone
only beta is raised (bursts: beta +29 %, gamma n.s.; power: beta +14 %, low gamma +3 %, high gamma +5 %, both n.s.).
Together with the quick check above (no late − correct difference in gamma in any region; no gamma start effect before the tone; no link between gamma and success),
the increase before the tone is specific to beta.

**Why:** a beta-specific result should not appear in the neighbouring gamma band measured the same way.

</div>

In [ ]:
fxr.run(fxb.gamma_step1)